# TRIBE v2 Experiments for Solace — Astronaut Mental Wellness App

Three experiments using Meta FAIR's TRIBE v2 brain encoder to validate design decisions:

1. **UI Design Comparison** — Compare multimodal vs. text-heavy screen layouts
2. **Nature Soundscape Testing** — Compare brain responses to different nature audio
3. **Multimodal vs. Unimodal** — Replicate the paper's core finding with our own content

**Setup:** Google Colab with GPU (Runtime > Change runtime type > T4 GPU)

In [ ]:
# Install (restart runtime after this cell)
!uv pip install "tribev2[plotting] @ git+https://github.com/facebookresearch/tribev2.git" opencv-python-headless

In [ ]:
from tribev2.demo_utils import TribeModel
from tribev2.plotting import PlotBrain
from pathlib import Path
import cv2
import numpy as np

CACHE = Path("./cache")
CACHE.mkdir(exist_ok=True)

model = TribeModel.from_pretrained("facebook/tribev2", cache_folder=CACHE)
plotter = PlotBrain(mesh="fsaverage5")

In [ ]:
import matplotlib.pyplot as plt
from PIL import Image

def image_to_video(image_path, output_path, duration=5, fps=24):
    """Convert a static image to a short video so TRIBE v2 can process it."""
    img = cv2.imread(str(image_path))
    h, w = img.shape[:2]
    writer = cv2.VideoWriter(str(output_path), cv2.VideoWriter_fourcc(*'mp4v'), fps, (w, h))
    for _ in range(duration * fps):
        writer.write(img)
    writer.release()
    return output_path


def predict_for_media(label, video_path=None, audio_path=None, text_path=None):
    """Run TRIBE v2 on any combination of video/audio/text. Returns averaged activation."""
    kwargs = {}
    if video_path: kwargs["video_path"] = video_path
    if audio_path: kwargs["audio_path"] = audio_path
    if text_path:  kwargs["text_path"] = text_path
    df = model.get_events_dataframe(**kwargs)
    preds, segments = model.predict(events=df)
    return preds.mean(axis=0), preds, segments


def predict_for_image(image_path, label):
    """Convenience: convert image to video, then predict."""
    video_path = CACHE / f"{label}.mp4"
    image_to_video(image_path, video_path)
    return predict_for_media(label, video_path=video_path)


def plot_and_save(activation, title, filename, cmap="fire", **kwargs):
    """Plot a single brain map and save it."""
    fig = plotter.plot_timesteps(
        activation[np.newaxis, :],
        cmap=cmap,
        norm_percentile=99,
        **kwargs,
    )
    fig.suptitle(title, fontsize=16, fontweight="bold", y=1.02)
    fig.savefig(CACHE / filename, dpi=150, bbox_inches="tight")
    plt.show()
    return fig


def plot_difference(avg_a, avg_b, label_a, label_b, filename):
    """Plot difference map between two activations."""
    diff = avg_a - avg_b
    print(f"Positive (warm) = more activation in {label_a}")
    print(f"Negative (cool) = more activation in {label_b}")
    return plot_and_save(diff, f"Difference: {label_a} vs {label_b}", filename, cmap="RdBu_r")

## Experiment 1: UI Design Comparison

Upload two screenshots — e.g. multimodal Restore Hub vs. text-heavy version.
Compare which brain regions each design activates.

In [ ]:
from google.colab import files

print("Upload your UI screenshots (2 images):")
uploaded = files.upload()

image_paths = []
for name in uploaded:
    path = CACHE / name
    path.write_bytes(uploaded[name])
    image_paths.append(path)
    print(f"  Saved: {path}")

print(f"\n{len(image_paths)} images ready.")

## Experiment 1: Results

In [ ]:
# Predict brain activation for each image
results = {}
for i, path in enumerate(image_paths):
    label = path.stem
    print(f"Processing: {label}")
    avg, preds, segments = predict_for_image(path, label)
    results[label] = {"avg": avg, "preds": preds, "segments": segments}

print(f"\nDone. {len(results)} predictions ready.")

In [ ]:
# Show input images side by side
fig, axes = plt.subplots(1, len(labels), figsize=(6 * len(labels), 4))
if len(labels) == 1: axes = [axes]
for ax, label, path in zip(axes, labels, image_paths):
    ax.imshow(Image.open(path))
    ax.set_title(label, fontsize=14, fontweight='bold')
    ax.axis('off')
plt.suptitle("Experiment 1: Input Screenshots", fontsize=16, y=1.02)
plt.tight_layout()
plt.savefig(CACHE / "exp1_inputs.png", dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Brain activation for each design
for label in labels:
    plot_and_save(results[label]["avg"], label, f"exp1_brain_{label}.png", vmin=0.5, alpha_cmap=(0, 0.2))

In [ ]:
# Difference map
if len(labels) == 2:
    plot_difference(results[labels[0]]["avg"], results[labels[1]]["avg"], labels[0], labels[1], "exp1_difference.png")

## Experiment 2: Nature Soundscape Comparison

Compare brain activation from different nature audio clips to validate which restoration content to include in the app.

Upload 2-4 short audio clips (.wav or .mp3, ~10 seconds each). For example:
- Rain sounds
- Ocean waves  
- Bird song
- Silence (control)

Free clips: [freesound.org](https://freesound.org) — search "rain ambience", "ocean waves", "forest birds"

In [ ]:
print("Upload nature audio clips (2-4 files, .wav or .mp3):")
uploaded_audio = files.upload()

audio_paths = []
for name in uploaded_audio:
    path = CACHE / name
    path.write_bytes(uploaded_audio[name])
    audio_paths.append(path)
    print(f"  Saved: {path}")

print(f"\n{len(audio_paths)} audio clips ready.")

In [ ]:
# Predict brain activation for each audio clip
audio_results = {}
for path in audio_paths:
    label = path.stem
    print(f"Processing: {label}")
    avg, preds, segments = predict_for_media(label, audio_path=path)
    audio_results[label] = avg

# Plot each
audio_labels = list(audio_results.keys())
for label in audio_labels:
    plot_and_save(audio_results[label], f"Audio: {label}", f"exp2_brain_{label}.png", vmin=0.5, alpha_cmap=(0, 0.2))

# Difference if exactly 2
if len(audio_labels) == 2:
    plot_difference(audio_results[audio_labels[0]], audio_results[audio_labels[1]], audio_labels[0], audio_labels[1], "exp2_difference.png")

## Experiment 3: Multimodal vs. Unimodal

The core finding from the TRIBE v2 paper (Figure 7C): audio + visual together activate more brain area than either alone.

Replicate this with your own content. Upload:
- A **nature video** clip (with audio) — e.g. a short clip of rain, ocean, forest
- The model will automatically run it 3 ways: video-only, audio-only, and combined

Free nature videos: [Pexels](https://www.pexels.com/search/videos/nature/) — download a short 10-15 second clip.

In [ ]:
print("Upload a nature video with audio (.mp4):")
uploaded_video = files.upload()

video_name = list(uploaded_video.keys())[0]
nature_video = CACHE / video_name
nature_video.write_bytes(uploaded_video[video_name])
print(f"Saved: {nature_video}")

In [ ]:
import subprocess

# Extract audio track from the video
audio_only = CACHE / "nature_audio_only.wav"
subprocess.run([
    "ffmpeg", "-y", "-i", str(nature_video),
    "-vn", "-acodec", "pcm_s16le", "-ar", "16000", str(audio_only)
], capture_output=True)

# Create a silent version of the video (video-only)
video_muted = CACHE / "nature_video_muted.mp4"
subprocess.run([
    "ffmpeg", "-y", "-i", str(nature_video),
    "-an", "-c:v", "copy", str(video_muted)
], capture_output=True)

print("Prepared:")
print(f"  Video + Audio (combined): {nature_video}")
print(f"  Video only (muted):       {video_muted}")
print(f"  Audio only:               {audio_only}")

In [ ]:
# Run all 3 conditions
print("=== Video + Audio (combined) ===")
avg_combined, _, _ = predict_for_media("combined", video_path=nature_video)

print("\n=== Video only (muted) ===")
avg_video, _, _ = predict_for_media("video_only", video_path=video_muted)

print("\n=== Audio only ===")
avg_audio, _, _ = predict_for_media("audio_only", audio_path=audio_only)

print("\nAll 3 conditions predicted.")

In [ ]:
# Plot all 3 conditions
plot_and_save(avg_video, "Video Only", "exp3_video_only.png", vmin=0.5, alpha_cmap=(0, 0.2))
plot_and_save(avg_audio, "Audio Only", "exp3_audio_only.png", vmin=0.5, alpha_cmap=(0, 0.2))
plot_and_save(avg_combined, "Video + Audio (Combined)", "exp3_combined.png", vmin=0.5, alpha_cmap=(0, 0.2))

In [ ]:
# The key comparison: Combined vs. best-of-unimodal
# This replicates Figure 7C from the paper
best_unimodal = np.maximum(avg_video, avg_audio)
multimodal_gain = avg_combined - best_unimodal

print("=== Multimodal Advantage ===")
print(f"Vertices where combined > best unimodal: {(multimodal_gain > 0).sum()} / {len(multimodal_gain)}")
print(f"  ({(multimodal_gain > 0).mean() * 100:.1f}% of cortex)")
print(f"Max gain: {multimodal_gain.max():.4f}")
print(f"Mean gain (where positive): {multimodal_gain[multimodal_gain > 0].mean():.4f}")

plot_and_save(
    multimodal_gain, 
    "Multimodal Advantage (Combined minus Best Unimodal)", 
    "exp3_multimodal_advantage.png", 
    cmap="hot",
    vmin=0,
)

print("\nIf the brain map lights up, it means audio+visual together")
print("activates MORE brain area than either modality alone.")
print("This is the core TRIBE v2 finding (Paper Fig. 7C) replicated with your content.")
print("\nDownload exp3_*.png files for your FigJam board.")